In [ ]:
import numpy as np
import itertools
import pickle
from tqdm import tqdm

class Tablero15:
    def __init__(self):
        # 9 posiciones (índices 0-8 representan los números 1-9)
        self.state = np.zeros(9)

    def valid_moves(self):
        # Retorna los índices (0-8) que aún tienen valor 0
        return [i for i in range(9) if self.state[i] == 0]

    def update(self, symbol, accion):
        if self.state[accion] != 0:
            raise ValueError('movimiento ilegal, numero ya elegido !')
        self.state[accion] = symbol

    def is_game_over(self):
        # Comprobar Jugador 1
        nums_p1 = [i + 1 for i, v in enumerate(self.state) if v == 1]
        for comb in itertools.combinations(nums_p1, 3):
            if sum(comb) == 15: return 1

        # Comprobar Jugador -1 (Agente/Humano)
        nums_p2 = [i + 1 for i, v in enumerate(self.state) if v == -1]
        for comb in itertools.combinations(nums_p2, 3):
            if sum(comb) == 15: return -1

        # Empate
        if len(self.valid_moves()) == 0:
            return 0

        # Seguir jugando
        return None

    def reset(self):
        self.state = np.zeros(9)

In [ ]:
class Agente15:
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}
        self.alpha = alpha
        self.positions = []
        self.prob_exp = prob_exp

    def reset(self):
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()
        # Exploracion epsilon-greedy
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))
            return valid_moves[ix]

        # Explotacion
        max_value = -1000
        best_action = valid_moves[0]
        for action in valid_moves:
            next_board = board.state.copy()
            next_board[action] = self.symbol
            next_state = str(next_board)
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)
            if value >= max_value:
                max_value = value
                best_action = action
        return best_action

    def update(self, board):
        self.positions.append(str(board.state))

    def reward(self, reward):
        # Actualizacion retrospectiva al final de la partida
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

class Juego15:
    def __init__(self, player1, player2):
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]
        self.board = Tablero15()

    def selfplay(self, rounds=100):
        wins = [0, 0]
        for i in tqdm(range(1, rounds + 1)):
            self.board.reset()
            for player in self.players:
                player.reset()
            game_over = False
            while not game_over:
                for player in self.players:
                    action = player.move(self.board)
                    self.board.update(player.symbol, action)
                    for p in self.players:
                        p.update(self.board)
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break
            self.reward()
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1
        return wins

    def reward(self):
        winner = self.board.is_game_over()
        if winner == 0:
            for player in self.players:
                player.reward(0.5)
        else:
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)
                else:
                    player.reward(0)

In [ ]:
# Entrenamos dos agentes jugando entre si
agent1 = Agente15(prob_exp=0.6)
agent2 = Agente15(prob_exp=0.5)
game = Juego15(agent1, agent2)

print("Entrenando...")
resultados = game.selfplay(50000)
print(f"Victorias Agente 1: {resultados[0]} | Victorias Agente 2: {resultados[1]}")

# Guardar la funcion de valor entrenada (como hizo tu docente)
with open('agente_juego15.pickle', 'wb') as handle:
    pickle.dump(agent1.value_function, handle)
print("¡Agente entrenado y guardado con éxito!")

In [ ]:
class AgenteInferencia15:
    def __init__(self, value_function, symbol=1):
        self.value_function = value_function
        self.symbol = symbol

    def move(self, board):
        valid_moves = board.valid_moves()
        max_value = -1000
        best_action = valid_moves[0]
        for action in valid_moves:
            next_board = board.state.copy()
            next_board[action] = self.symbol
            next_state = str(next_board)
            value = self.value_function.get(next_state, 0)
            if value >= max_value:
                max_value = value
                best_action = action
        return best_action

def jugar_15(agente_empieza=True):
    # Cargar el conocimiento entrenado
    with open('agente_juego15.pickle', 'rb') as handle:
        funcion_de_valor = pickle.load(handle)

    board = Tablero15()
    agente = AgenteInferencia15(funcion_de_valor, symbol=1)
    simbolo_humano = -1

    print("--- EL JUEGO DE 15 ---")
    print("Objetivo: Coleccionar 3 números que sumen EXACTAMENTE 15.")
    print("Agente: 1 | Humano: -1\\n")

    turno_agente = agente_empieza
    while board.is_game_over() is None:
        disponibles = [i+1 for i in board.valid_moves()]
        if turno_agente:
            accion = agente.move(board)
            board.update(agente.symbol, accion)
            print(f"La IA eligió el número: {accion + 1}")
        else:
            numeros_p1 = [i+1 for i, v in enumerate(board.state) if v == 1]
            numeros_p2 = [i+1 for i, v in enumerate(board.state) if v == -1]
            print(f"Tus números: {numeros_p2} | Números de IA: {numeros_p1}")

            while True:
                try:
                    entrada = int(input(f"Tu turno. Elige uno de {disponibles}: "))
                    if entrada in disponibles:
                        accion = entrada - 1
                        break
                    print("Número no disponible.")
                except ValueError:
                    print("Ingresa un número válido.")
            board.update(simbolo_humano, accion)

        turno_agente = not turno_agente

    resultado = board.is_game_over()
    if resultado == 1: print("\\n¡La IA gana!")
    elif resultado == -1: print("\\n¡Ganaste!")
    else: print("\\n¡Empate!")

# Ejecuta para jugar:
jugar_15(agente_empieza=False)